# HMDA 2024: vì sao cache làm Spark chậm hơn ở kích thước lớn?

Thí nghiệm hiệu năng chính cho thấy từ ×4 trở lên, chạy trên bảng đã cache lại chậm hơn đọc thẳng Parquet.
Notebook này kiểm hai giả thuyết trên đúng bảng 15 cột đó:
1. **Cột:** đọc Parquet chỉ lấy các cột truy vấn cần (column pruning), còn cache giữ cả 15 cột.
   Kiểm bằng cách cache riêng các cột W1 dùng.
2. **Bộ nhớ:** bảng cache không vừa vùng storage memory của Spark nên bị đẩy xuống đĩa.
   Kiểm bằng thông tin lưu trữ của từng RDD đã cache (kích thước trong RAM, trên đĩa, số partition đã cache).

Dữ liệu dựng bằng đúng hai ô đầu của `perf_pipeline.py` (trên Kaggle: hai ô đó được chép ngay sau ô này).
Usage (ngoài Kaggle): `HMDA_INPUT=... python perf_cache_diagnosis.py`

In [1]:
import os
import sys

os.environ.setdefault("PERF_SIZES", "100%,x4,x8,x16")
IN_NOTEBOOK = "ipykernel" in sys.modules
if not IN_NOTEBOOK:   # chạy hai ô đầu (cài đặt, dựng dữ liệu) của perf_pipeline.py
    _src = open(os.path.join(os.path.dirname(os.path.abspath(__file__)), "perf_pipeline.py"), encoding="utf-8").read()
    _cells = [c for c in _src.split("# %%") if not c.startswith(" [markdown]")]
    for _c in _cells[1:3]:
        exec(compile(_c, "perf_pipeline.py", "exec"))

---
# Hai ô đầu của `perf_pipeline.py`: cài đặt và dựng dữ liệu

In [2]:
# ===== 0. Cài đặt =====
import os, sys, glob, json, time, math, shutil, subprocess, platform, textwrap, zipfile, urllib.request
import numpy as np
import pandas as pd
import pyarrow as pa
import pyarrow.compute as pc
import pyarrow.parquet as pq
import yaml
import matplotlib

try:
    get_ipython  # noqa: F821
    IN_NOTEBOOK = True
except NameError:
    IN_NOTEBOOK = False
    matplotlib.use("Agg")
import matplotlib.pyplot as plt

try:
    import psutil
except ImportError:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "psutil"], check=True); import psutil
try:
    import pyspark  # noqa: F401
except ImportError:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "pyspark"], check=True)


def find_file(pattern, required=True):
    roots = [os.environ["HMDA_INPUT"]] if os.environ.get("HMDA_INPUT") else ["/kaggle/input"]
    hits = sorted(h for r_ in roots for h in glob.glob(os.path.join(r_, "**", pattern), recursive=True))
    assert hits or not required, f"Không thấy {pattern} trong {roots}"
    return hits[0] if hits else None


RAW_SRC = find_file("hmda_2024_raw.parquet")
R = yaml.safe_load(open(find_file("rules.yaml"), encoding="utf-8"))
assert float(R["version"]) >= 1.1
TS_PATH = find_file("*public_ts*.csv", required=False) or find_file("*public_ts*.zip", required=False)
OUTROOT = "/kaggle/working" if os.path.isdir("/kaggle/working") else os.environ.get("HMDA_OUT", "output")
RES, FIG = f"{OUTROOT}/results/perf", f"{OUTROOT}/figures/perf"
WORK = os.environ.get("HMDA_WORK", "/tmp/hmda_perf")
for d in (RES, FIG, WORK, f"{WORK}/data", f"{WORK}/logs"):
    os.makedirs(d, exist_ok=True)
PF = R["performance"]
WARMUP, RUNS = int(os.environ.get("PERF_WARMUP", PF["warmup_runs"])), int(os.environ.get("PERF_RUNS", PF["timed_runs"]))
def container_ram_bytes():
    """RAM dùng được: min(RAM máy, giới hạn cgroup của container) vì Kaggle giới hạn bằng cgroup (30 GiB)."""
    tot = psutil.virtual_memory().total
    for p_ in ("/sys/fs/cgroup/memory.max", "/sys/fs/cgroup/memory/memory.limit_in_bytes"):
        try:
            v_ = open(p_).read().strip()
            if v_.isdigit():
                tot = min(tot, int(v_))
        except OSError:
            pass
    return tot


TOTAL_RAM = container_ram_bytes()
BUDGET = float(os.environ.get("PERF_BUDGET_GB", PF["memory"]["budget_share_of_ram"] * TOTAL_RAM / 1e9)) * 1e9
SIZES = os.environ.get("PERF_SIZES", ",".join(PF["sizes"])).split(",")
ENV = {"python": sys.version.split()[0], "platform": platform.platform(), "cpu_logical": os.cpu_count(),
       "ram_total_gb": round(TOTAL_RAM / 1e9, 1), "memory_budget_gb": round(BUDGET / 1e9, 1), "pandas": pd.__version__,
       "pyarrow": pa.__version__, "pyspark": __import__("pyspark").__version__, "rules_version": str(R["version"])}
if TS_PATH and TS_PATH.endswith(".zip"):
    with zipfile.ZipFile(TS_PATH) as z_:
        nm_ = [n for n in z_.namelist() if n.lower().endswith((".csv", ".txt"))][0]
        TS_PATH = z_.extract(nm_, WORK)
print(json.dumps(ENV, indent=1)); print("TS:", TS_PATH)

{
 "python": "3.13.18",
 "platform": "Linux-6.18.55+-x86_64-with-glibc2.39",
 "cpu_logical": 4,
 "ram_total_gb": 32.2,
 "memory_budget_gb": 29.0,
 "pandas": "2.3.3",
 "pyarrow": "23.0.1",
 "pyspark": "4.0.4",
 "rules_version": "1.1"
}
TS: None


## 1. Tạo bảng đo `perf_core` và các kích thước (không tính giờ)
Cùng định nghĩa với hai notebook phân tích: lọc core, usable, ô rủi ro 6 × 5 × 3, p_ext theo ô mở rộng (fallback ô rủi ro).

In [3]:
t0 = time.time()
V, D, L = R["variables"], R["variables"]["debt_to_income_ratio"], R["variables"]["combined_loan_to_value_ratio"]
FILT = R["samples"]["core_filters"]
cols = sorted({f["column"] for f in FILT} | {"row_id", "lei", "state_code", "county_code", "derived_msa_md", "action_taken",
                                             "loan_type", "loan_purpose", "loan_amount", "income", "debt_to_income_ratio",
                                             "combined_loan_to_value_ratio"})
parts = []
for b in pq.ParquetFile(RAW_SRC).iter_batches(batch_size=500_000, columns=cols):
    tb = pa.Table.from_batches([b])
    for f in FILT:
        col = tb[f["column"]]
        tb = tb.filter(pc.is_in(col, pa.array(f["in"])) if "in" in f else pc.equal(col, f["equals"]))
    parts.append(tb)
c = pa.concat_tables(parts).to_pandas(); del parts


def num(s):
    return pd.to_numeric(s.where(~s.isin(["NA", "Exempt", ""])), errors="coerce")


def cut(x, spec, cuts=None):
    rs = lambda v: cuts[v] if isinstance(v, str) else float(v)
    return pd.cut(x, [rs(spec[0]["lo"])] + [rs(b_["hi"]) for b_ in spec], labels=[b_["label"] for b_ in spec], right=True)


dti = c["debt_to_income_ratio"].str.replace("%", "", regex=False).str.replace(" ", "", regex=False).map(
    {str(k): v for k, v in D["ordinal_map"].items()})
cltv, inc, amt = num(c["combined_loan_to_value_ratio"]), num(c["income"]), num(c["loan_amount"])
cltv_ok = (cltv > L["valid_range"]["gt"]) & (cltv <= L["valid_range"]["le"])
P = pd.DataFrame({
    "row_id": c["row_id"].astype("int64"), "lei": c["lei"], "state_code": c["state_code"], "county_code": c["county_code"],
    "msa": c["derived_msa_md"], "action_taken": c["action_taken"].astype("int8"), "loan_type": c["loan_type"].astype("int8"),
    "loan_purpose": c["loan_purpose"].astype("int16"), "loan_amount": amt})
P["county_valid"] = P["county_code"].str.fullmatch(R["geography"]["county_code_format"].strip("^$")).fillna(False)
P["decisioned"] = P["action_taken"].isin([1, 2, 3]); P["denied"] = P["action_taken"] == 3
P["usable"] = P["decisioned"] & dti.notna() & cltv_ok & (inc > 0)
P["y"] = P["denied"].astype("float64")
lti = np.where(inc > 0, amt / (1000 * inc), np.nan)
B = pd.DataFrame({"d": cut(dti, D["bands"]), "c": cut(cltv.where(cltv_ok), L["bands"]),
                  "l": cut(pd.Series(lti), V["loan_to_income"]["bands"], V["loan_to_income"]["cut_exact"]),
                  "t": P["loan_type"], "p": P["loan_purpose"], "y": P["y"]})[P["usable"].to_numpy()]
g_ = B.groupby(["d", "c", "l", "t", "p"], observed=True)["y"]
pe = np.where(g_.transform("size") >= R["extended_cells"]["min_n"], g_.transform("mean"),
              B.groupby(["d", "c", "l"], observed=True)["y"].transform("mean"))
P["p_ext"] = np.nan
P.loc[P["usable"].to_numpy(), "p_ext"] = pe
P = P[PF["columns"]]
del c, B, g_, dti, cltv, inc, amt
S_ = R["sampling"]
bucket = ((P["row_id"] * S_["hash_multiplier"]) % S_["hash_modulus"]) % S_["buckets"]
DATA = {}
for sz in SIZES:
    path = f"{WORK}/data/{sz.replace('%', 'pct')}"
    if os.path.exists(path):
        shutil.rmtree(path)
    os.makedirs(path)
    if sz.endswith("%"):
        sub = pa.Table.from_pandas(P[bucket < int(sz[:-1])], preserve_index=False)
        pq.write_table(sub, f"{path}/part-0.parquet"); rows = sub.num_rows
    else:
        k = int(sz[1:]); base = pa.Table.from_pandas(P, preserve_index=False)
        for j in range(k):
            pq.write_table(base.set_column(base.schema.get_field_index("row_id"), "row_id",
                                           pc.add(base["row_id"], j * S_["replicas"]["row_id_offset"])), f"{path}/part-{j}.parquet")
        rows = base.num_rows * k
    DATA[sz] = {"path": path, "rows": int(rows), "disk_mb": round(sum(os.path.getsize(f) for f in glob.glob(f"{path}/*")) / 1e6, 1)}
del P, bucket
json.dump(DATA, open(f"{WORK}/data/sizes.json", "w"))
print(pd.DataFrame(DATA).T.to_string()); print(f"chuẩn bị dữ liệu: {time.time() - t0:.0f}s")

                            path      rows disk_mb
100%  /tmp/hmda_perf/data/100pct   5775325    63.2
x4        /tmp/hmda_perf/data/x4  23101300   252.8
x8        /tmp/hmda_perf/data/x8  46202600   505.5
x16      /tmp/hmda_perf/data/x16  92405200  1011.1
chuẩn bị dữ liệu: 155s


## Đo trên Spark

In [4]:
import statistics
import time

import pandas as pd
from pyspark.sql import SparkSession, functions as F

spark = (SparkSession.builder.master("local[*]").appName("hmda-cache-diag")
         .config("spark.driver.memory", f"{max(2, int(TOTAL_RAM / 1e9 * 0.6))}g")
         .config("spark.sql.shuffle.partitions", str(max(8, 2 * (os.cpu_count() or 2))))
         .config("spark.local.dir", f"{WORK}/spark_local").getOrCreate())
spark.sparkContext.setLogLevel("ERROR")
jsc = spark.sparkContext._jsc.sc()
storage_max = None
try:
    ms = jsc.getExecutorMemoryStatus()
    it = ms.values().iterator()
    storage_max = 0
    while it.hasNext():
        storage_max += it.next()._1()
except Exception as e:  # noqa: BLE001
    print("không đọc được bộ nhớ storage:", repr(e)[:100])
print("storage memory tối đa (GB):", None if storage_max is None else round(storage_max / 1e9, 2))

W1_COLS = ["decisioned", "loan_type", "loan_purpose", "y", "loan_amount"]


def w1(df):
    g = df.filter("decisioned").groupBy("loan_type", "loan_purpose").agg(
        F.count(F.lit(1)).alias("n"), F.avg("y").alias("rate"), F.avg("loan_amount").alias("mean_amount"))
    return g.collect()


def timed(fn, runs=3):
    fn()   # làm nóng
    ts = []
    for _ in range(runs):
        t = time.perf_counter(); fn(); ts.append(time.perf_counter() - t)
    return statistics.median(ts)


def storage():
    mem = disk = cached = total = 0
    for r in jsc.getRDDStorageInfo():
        mem += r.memSize(); disk += r.diskSize(); cached += r.numCachedPartitions(); total += r.numPartitions()
    return mem / 1e9, disk / 1e9, cached, total


rows = []
for sz in SIZES:
    path = DATA[sz]["path"]
    base = spark.read.parquet(path)
    r = {"size": sz, "rows": DATA[sz]["rows"], "parquet_gb": DATA[sz]["disk_mb"] / 1e3}
    r["w1_nocache_s"] = timed(lambda: w1(spark.read.parquet(path)))
    for label, df in [("full", base), ("w1cols", base.select(*W1_COLS))]:
        df = df.cache()
        t = time.perf_counter(); df.count(); r[f"cache_{label}_build_s"] = time.perf_counter() - t
        mem, disk, cached, total = storage()
        r.update({f"cache_{label}_mem_gb": mem, f"cache_{label}_disk_gb": disk,
                  f"cache_{label}_partitions_cached": cached, f"cache_{label}_partitions": total})
        r[f"w1_cache_{label}_s"] = timed(lambda: w1(df))
        df.unpersist(blocking=True)
    rows.append(r)
    print({k: (round(v, 3) if isinstance(v, float) else v) for k, v in r.items()})
spark.stop()

D = pd.DataFrame(rows)
D["storage_max_gb"] = None if storage_max is None else storage_max / 1e9
OUTROOT = "/kaggle/working" if os.path.isdir("/kaggle/working") else os.environ.get("HMDA_OUT", "output")
os.makedirs(f"{OUTROOT}/results/perf", exist_ok=True)
D.to_csv(f"{OUTROOT}/results/perf/perf_cache_diagnosis.csv", index=False)
pd.set_option("display.width", 250)
print(D.round(3).to_string(index=False))

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/11/16 11:57:21 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/11/16 11:57:21 WARN SparkConf: Note that spark.local.dir will be overridden by the value set by the cluster manager (via SPARK_LOCAL_DIRS in standalone/kubernetes and LOCAL_DIRS in YARN).


storage memory tối đa (GB): 12.05


{'size': '100%', 'rows': 5775325, 'parquet_gb': 0.063, 'w1_nocache_s': 0.991, 'cache_full_build_s': 14.025, 'cache_full_mem_gb': 0.17, 'cache_full_disk_gb': 0.0, 'cache_full_partitions_cached': 4, 'cache_full_partitions': 4, 'w1_cache_full_s': 0.698, 'cache_w1cols_build_s': 2.681, 'cache_w1cols_mem_gb': 0.075, 'cache_w1cols_disk_gb': 0.0, 'cache_w1cols_partitions_cached': 4, 'cache_w1cols_partitions': 4, 'w1_cache_w1cols_s': 0.628}


{'size': 'x4', 'rows': 23101300, 'parquet_gb': 0.253, 'w1_nocache_s': 1.282, 'cache_full_build_s': 40.083, 'cache_full_mem_gb': 0.586, 'cache_full_disk_gb': 0.0, 'cache_full_partitions_cached': 4, 'cache_full_partitions': 4, 'w1_cache_full_s': 1.637, 'cache_w1cols_build_s': 7.633, 'cache_w1cols_mem_gb': 0.257, 'cache_w1cols_disk_gb': 0.0, 'cache_w1cols_partitions_cached': 4, 'cache_w1cols_partitions': 4, 'w1_cache_w1cols_s': 1.594}


{'size': 'x8', 'rows': 46202600, 'parquet_gb': 0.505, 'w1_nocache_s': 2.084, 'cache_full_build_s': 78.154, 'cache_full_mem_gb': 1.282, 'cache_full_disk_gb': 0.0, 'cache_full_partitions_cached': 4, 'cache_full_partitions': 4, 'w1_cache_full_s': 3.118, 'cache_w1cols_build_s': 14.871, 'cache_w1cols_mem_gb': 0.565, 'cache_w1cols_disk_gb': 0.0, 'cache_w1cols_partitions_cached': 4, 'cache_w1cols_partitions': 4, 'w1_cache_w1cols_s': 2.973}


{'size': 'x16', 'rows': 92405200, 'parquet_gb': 1.011, 'w1_nocache_s': 3.84, 'cache_full_build_s': 155.861, 'cache_full_mem_gb': 2.564, 'cache_full_disk_gb': 0.0, 'cache_full_partitions_cached': 8, 'cache_full_partitions': 8, 'w1_cache_full_s': 6.084, 'cache_w1cols_build_s': 29.551, 'cache_w1cols_mem_gb': 1.13, 'cache_w1cols_disk_gb': 0.0, 'cache_w1cols_partitions_cached': 8, 'cache_w1cols_partitions': 8, 'w1_cache_w1cols_s': 5.907}
size     rows  parquet_gb  w1_nocache_s  cache_full_build_s  cache_full_mem_gb  cache_full_disk_gb  cache_full_partitions_cached  cache_full_partitions  w1_cache_full_s  cache_w1cols_build_s  cache_w1cols_mem_gb  cache_w1cols_disk_gb  cache_w1cols_partitions_cached  cache_w1cols_partitions  w1_cache_w1cols_s  storage_max_gb
100%  5775325       0.063         0.991              14.025              0.170                 0.0                             4                      4            0.698                 2.681                0.075                   0.0    

In [5]:
# In các file kết quả (nhỏ) kèm md5 để lưu lại vào repo
import glob, hashlib, os
for p in sorted(glob.glob("/kaggle/working/results/perf/*", recursive=True)):
    if os.path.isfile(p) and os.path.getsize(p) < 200_000 and not p.endswith((".parquet", ".png", ".npz")):
        b = open(p, "rb").read()
        print("@@@@ FILE", os.path.relpath(p, "/kaggle/working"), hashlib.md5(b).hexdigest(), len(b))
        print(b.decode("utf-8"))
        print("@@@@ END")

@@@@ FILE results/perf/perf_cache_diagnosis.csv 56ec6c5b970ca062f2d9c4048140ef2c 967
size,rows,parquet_gb,w1_nocache_s,cache_full_build_s,cache_full_mem_gb,cache_full_disk_gb,cache_full_partitions_cached,cache_full_partitions,w1_cache_full_s,cache_w1cols_build_s,cache_w1cols_mem_gb,cache_w1cols_disk_gb,cache_w1cols_partitions_cached,cache_w1cols_partitions,w1_cache_w1cols_s,storage_max_gb
100%,5775325,0.0632,0.9906259000000546,14.02526689900003,0.16992128,0.0,4,4,0.697994036999944,2.6814210509999157,0.074704672,0.0,4,4,0.6276794059999702,12.051913113
x4,23101300,0.2528,1.2816371280000567,40.08289656299996,0.586477728,0.0,4,4,1.6370279139999866,7.6329990439999165,0.2573696,0.0,4,4,1.5944004120000272,12.051913113
x8,46202600,0.5055,2.084003562000021,78.15406848999999,1.281759744,0.0,4,4,3.1179992879999645,14.871375767000018,0.565001632,0.0,4,4,2.9732294430000366,12.051913113
x16,92405200,1.0111,3.8400631950000843,155.86084647300004,2.563519488,0.0,8,8,6.084418563000099,29.550686213000063